# Лабораторная работа №15: Speech-to-Text и Text-to-Speech (Whisper, Silero)

**Студент:** [ФИО]  
**Группа:** [Номер группы]

## Цель работы
Изучить технологии обработки речи:
- Распознавание речи (Speech-to-Text) с помощью Whisper
- Синтез речи (Text-to-Speech) с использованием Silero и XTTS
- Создание голосового интерфейса для приложений

## Теоретические сведения

### Speech-to-Text (STT)
Распознавание речи преобразует аудиосигнал в текст. Современные подходы:
- **Encoder-Decoder архитектуры** — трансформеры для последовательностей
- **CTC (Connectionist Temporal Classification)** — выравнивание аудио и текста
- **End-to-End модели** — прямое преобразование без промежуточных этапов

### Whisper (OpenAI)
Whisper — мультимодальная модель для распознавания речи:
- Обучена на 680,000 часах размеченных данных
- Поддержка 99 языков, включая русский
- Распознавание + перевод + определение языка
- Устойчивость к шумам и акцентам

### Text-to-Speech (TTS)
Синтез речи создает естественное звучание из текста:
- **Конкатенативный синтез** — сборка из записанных фрагментов
- **Параметрический синтез** — генерация через vocoder
- **Neural TTS** — Tacotron, FastSpeech, VITS

### Silero TTS
Silero — легковесные модели для синтеза речи:
- Поддержка русского и английского языков
- Быстрый инференс на CPU
- Несколько голосов и эмоциональных окрасок

### XTTS (Coqui)
XTTS — продвинутая TTS модель:
- Клонирование голоса по образцу (voice cloning)
- Высокое качество звучания
- Мультиязычность

## Задание

### Базовый уровень
1. Установите необходимые библиотеки
2. Загрузите модель Whisper для распознавания речи
3. Распознайте речь из аудиофайла или микрофона
4. Реализуйте простой TTS с Silero

### Продвинутый уровень
1. Создайте пайплайн STT → LLM → TTS для голосового ассистента
2. Реализуйте клонирование голоса с XTTS
3. Добавьте обработку ошибок и повторные попытки

### Индивидуальный уровень
1. Оптимизируйте модели для работы в реальном времени
2. Исследуйте влияние качества аудио на точность распознавания
3. Создайте многопользовательский голосовой интерфейс

## Ход работы

In [ ]:
# Установка необходимых библиотек
!pip install -q openai-whisper torch torchaudio silero-tts TTS pydub

In [ ]:
import torch
import whisper
from IPython.display import Audio, display
import warnings
warnings.filterwarnings('ignore')

# Проверка доступности GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используемое устройство: {device}")

### Часть 1: Speech-to-Text с Whisper

In [ ]:
# Шаг 1: Загрузка модели Whisper
# Доступные размеры: tiny, base, small, medium, large
model_size = "base"  # Для Colab рекомендуется base или small
whisper_model = whisper.load_model(model_size, device=device)

print(f"Модель Whisper {model_size} загружена")
print(f"Параметры модели: {sum(p.numel() for p in whisper_model.parameters()):,}")

In [ ]:
# Шаг 2: Создание тестового аудио (синтезированного)
# В реальном проекте используйте свои аудиофайлы

# Для демонстрации скачаем пример аудио
import urllib.request
import os

audio_url = "https://github.com/openai/whisper/raw/main/tests/audio/jfk.flac"
audio_path = "test_audio.flac"

if not os.path.exists(audio_path):
    print("Загрузка тестового аудио...")
    urllib.request.urlretrieve(audio_url, audio_path)
    print("Аудио загружено")
else:
    print("Аудио файл уже существует")

# Воспроизведение аудио
display(Audio(audio_path))

In [ ]:
# Шаг 3: Распознавание речи
transcription_options = {
    "language": "en",  # Язык аудио (или None для автоопределения)
    "task": "transcribe",  # transcribe или translate
    "verbose": False
}

result = whisper_model.transcribe(audio_path, **transcription_options)

print("=== Результат распознавания ===")
print(f"Язык: {result['language']}")
print(f"Текст: {result['text']}")
print(f"\nДетали по сегментам:")
for segment in result.get('segments', []):
    print(f"  [{segment['start']:.2f}s - {segment['end']:.2f}s]: {segment['text']}")

In [ ]:
# Шаг 4: Распознавание с различными параметрами
def compare_whisper_models(audio_file, sizes=["tiny", "base"]):
    """
    Сравнение результатов разных моделей Whisper
    """
    results = {}
    
    for size in sizes:
        print(f"\nЗагрузка модели {size}...")
        model = whisper.load_model(size, device=device)
        
        import time
        start = time.time()
        result = model.transcribe(audio_file, language="en")
        elapsed = time.time() - start
        
        results[size] = {
            "text": result["text"],
            "time": elapsed,
            "language": result["language"]
        }
        
        print(f"  Время: {elapsed:.2f}s")
        print(f"  Текст: {result['text'][:100]}...")
    
    return results

# Сравнение моделей (может занять время)
# model_comparison = compare_whisper_models(audio_path)

### Часть 2: Text-to-Speech с Silero

In [ ]:
# Шаг 5: Загрузка модели Silero TTS
torch.hub.download_url_to_file(
    'https://models.silero.ai/models/tts/ru/v3_1_ru.pth',
    'ru_model.pth'
)

model, example_text = torch.hub.load(
    repo_or_dir='snakers4/silero-models',
    model='silero_tts',
    language='ru',
    speaker='v3_1'
)

print("Модель Silero TTS загружена")
print(f"Пример текста: {example_text[:50]}...")

In [ ]:
# Шаг 6: Синтез речи
text_to_speak = "Привет! Это демонстрация синтеза речи с помощью Silero."

# Выбор голоса (доступные зависят от версии модели)
speaker = "kseniya"  # или "aidar", "baya", etc.
sample_rate = 48000  # 24000 или 48000

audio_tensor = model.apply_tts(
    text=text_to_speak,
    speaker=speaker,
    sample_rate=sample_rate
)

# Сохранение и воспроизведение
import scipy.io.wavfile as wavfile
output_path = "synthesized_speech.wav"
wavfile.write(output_path, sample_rate, audio_tensor.cpu().numpy())

print(f"Аудио сохранено в {output_path}")
display(Audio(output_path))

In [ ]:
# Шаг 7: Синтез с разными голосами
speakers = ["kseniya", "aidar"]  # Доступные голоса
texts = [
    "Добрый день! Как ваши дела?",
    "Это тест системы синтеза речи."
]

for i, text in enumerate(texts):
    speaker = speakers[i % len(speakers)]
    
    audio = model.apply_tts(
        text=text,
        speaker=speaker,
        sample_rate=sample_rate
    )
    
    path = f"speech_{speaker}.wav"
    wavfile.write(path, sample_rate, audio.cpu().numpy())
    
    print(f"\nГолос: {speaker}")
    print(f"Текст: {text}")
    display(Audio(path))

### Часть 3: Голосовой ассистент (STT → LLM → TTS)

In [ ]:
# Шаг 8: Простой голосовой ассистент
from transformers import pipeline

# Загрузка небольшой модели для ответов
print("Загрузка модели для генерации ответов...")
qa_pipeline = pipeline(
    "text-generation",
    model="google/flan-t5-small",
    device=device if device != "cpu" else -1
)

def generate_response(question: str, max_length: int = 50) -> str:
    """
    Генерация ответа на вопрос
    """
    prompt = f"Answer the question concisely: {question}"
    result = qa_pipeline(prompt, max_length=max_length, num_return_sequences=1)
    return result[0]['generated_text'].replace(prompt, '').strip()

print("Ассистент готов")

In [ ]:
# Шаг 9: Полный пайплайн голосового ассистента
def voice_assistant(audio_input_path: str, output_path: str = "response.wav"):
    """
    Полный пайплайн: STT → LLM → TTS
    """
    print("\n=== Голосовой ассистент ===")
    
    # 1. Распознавание речи
    print("1. Распознавание речи...")
    stt_result = whisper_model.transcribe(audio_input_path, language="en")
    user_question = stt_result["text"]
    print(f"   Вопрос: {user_question}")
    
    # 2. Генерация ответа
    print("2. Генерация ответа...")
    response_text = generate_response(user_question)
    print(f"   Ответ: {response_text}")
    
    # 3. Синтез речи (на английском для простоты)
    print("3. Синтез речи...")
    # Для английского можно использовать другую модель или перевод
    print("   (В реальном проекте используйте мультиязычную TTS)")
    
    return {
        "question": user_question,
        "answer": response_text
    }

# Пример использования (раскомментировать при наличии аудио)
# result = voice_assistant(audio_path)

### Часть 4: Продвинутый TTS с XTTS (опционально)

In [ ]:
# Шаг 10: Информация о XTTS для клонирования голоса
print("""
XTTS v2 от Coqui AI поддерживает:
- Клонирование голоса по 6-секундному образцу
- 17 языков включая русский
- Высокое качество синтеза

Пример кода для XTTS:

from TTS.api import TTS

# Инициализация
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=True if device=='cuda' else False)

# Клонирование голоса
tts.tts_to_file(
    text="Привет! Это клонированный голос.",
    speaker_wav="path/to/reference_voice.wav",
    language="ru",
    file_path="cloned_output.wav"
)

Примечание: XTTS требует больше ресурсов и может не работать на бесплатном Colab.
""")

In [ ]:
# Шаг 11: Оценка качества распознавания
def evaluate_stt(reference_text: str, recognized_text: str) -> dict:
    """
    Простая оценка качества STT через Word Error Rate (WER)
    """
    # Нормализация текста
    def normalize(text):
        return text.lower().strip().split()
    
    ref_words = normalize(reference_text)
    rec_words = normalize(recognized_text)
    
    # Простой подсчет совпадений
    correct = sum(1 for w in rec_words if w in ref_words)
    total = len(ref_words)
    
    accuracy = correct / total if total > 0 else 0
    
    return {
        "accuracy": accuracy,
        "correct_words": correct,
        "total_words": total
    }

# Пример оценки
reference = "And so my fellow Americans ask not what your country can do for you"
recognized = result["text"]

metrics = evaluate_stt(reference, recognized)
print(f"\n=== Оценка качества STT ===")
print(f"Точность: {metrics['accuracy']:.2%}")
print(f"Верно распознано слов: {metrics['correct_words']} из {metrics['total_words']}")

## Контрольные вопросы

1. Какие архитектуры используются в современных моделях STT?
2. Чем отличаются модели Whisper разных размеров (tiny, base, large)?
3. Как работает механизм внимания в трансформерах для обработки аудио?
4. Какие преимущества у нейронного синтеза речи перед традиционными методами?
5. Как реализовать клонирование голоса и какие этические вопросы это поднимает?

## Выводы

В данной лабораторной работе были изучены технологии обработки речи:
- Освоено распознавание речи с помощью Whisper (OpenAI)
- Реализован синтез речи через Silero TTS
- Создан прототип голосового ассистента (STT → LLM → TTS)
- Изучены возможности клонирования голоса с XTTS
- Проанализированы метрики качества распознавания

Голосовые интерфейсы становятся стандартом для взаимодействия с приложениями, а современные модели обеспечивают высокое качество как распознавания, так и синтеза речи.